# Description

Here, we aggregate the results of Korns experiment.

In [1]:
import pandas as pd
import sympy as sp
from functools import lru_cache
import math

# -------------------------
# User input: list your CSV files here
# -------------------------
csv_files = [
    "korns_pysr_benchmark_results.csv",
    "korns_sindy_benchmark_results.csv",
    "korns_eql_div_benchmark_results.csv",
    "korns_complexeql_benchmark_results.csv",
]

# -------------------------
# SymPy parsing + node count
# -------------------------
_symbols = {f"x{i}": sp.Symbol(f"x{i}") for i in range(100)}

_locals = {
    **_symbols,
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "tanh": sp.tanh,
    "exp": sp.exp,
    "log": sp.log,
    "sqrt": sp.sqrt,
    "Abs": sp.Abs,
    "abs": sp.Abs,
}

@lru_cache(maxsize=200_000)
def _parse_expr(expr_str: str):
    if expr_str is None:
        return None
    s = str(expr_str).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    try:
        return sp.sympify(s, locals=_locals)
    except Exception:
        return None

@lru_cache(maxsize=200_000)
def _node_count(expr_str: str):
    expr = _parse_expr(expr_str)
    if expr is None:
        return None
    try:
        return sum(1 for _ in sp.preorder_traversal(expr))
    except Exception:
        return None

# -------------------------
# Formatting helpers (stdout, no LaTeX)
# -------------------------
def _is_finite(x) -> bool:
    try:
        return x is not None and math.isfinite(float(x))
    except Exception:
        return False

def _fmt_pm_sci10(mean, std, decimals=1) -> str:
    # Always returns: (m+-s)*10^e  (or 0.0)
    if not _is_finite(mean):
        return "0.0"
    m = float(mean)
    s = float(std) if _is_finite(std) else 0.0

    if m == 0.0 and s == 0.0:
        return "0.0"

    a = max(abs(m), abs(s))
    if a == 0.0:
        return "0.0"

    exp10 = int(math.floor(math.log10(a)))
    scale = 10.0 ** exp10
    m_s = m / scale
    s_s = s / scale
    return f"({m_s:.{decimals}f}+-{s_s:.{decimals}f})*10^{exp10}"

def _pid_to_korns(pid: str) -> str:
    s = str(pid).strip()
    if s.startswith("P") and s[1:].isdigit():
        return f"Korns-{int(s[1:])}"
    return s

def _print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(str(cell)))

    def fmt_row(row):
        return "  ".join(str(cell).ljust(widths[j]) for j, cell in enumerate(row))

    print(fmt_row(headers))
    print("  ".join("-" * w for w in widths))
    for row in rows:
        print(fmt_row(row))

# -------------------------
# Aggregate + print as table-like stdout
# -------------------------
for path in csv_files:
    df = pd.read_csv(path)

    df["mse_train"] = pd.to_numeric(df["mse_train"], errors="coerce")
    df["mse_test"]  = pd.to_numeric(df["mse_test"],  errors="coerce")

    df["node_count"] = df["expr_str"].astype(str).map(_node_count)
    df["node_count"] = pd.to_numeric(df["node_count"], errors="coerce")

    agg = (
        df.groupby(["pid", "algo"], dropna=False)
          .agg(
              n_runs=("run_id", "count"),
              mse_train_mean=("mse_train", "mean"),
              mse_train_std=("mse_train", "std"),
              mse_test_mean=("mse_test", "mean"),
              mse_test_std=("mse_test", "std"),
              node_count_mean=("node_count", "mean"),
              node_count_std=("node_count", "std"),
          )
          .reset_index()
          .sort_values(["pid", "algo"])
    )

    algo_name = str(agg["algo"].iloc[0]) if len(agg) else "algo"

    rows = []
    for _, r in agg.iterrows():
        dataset = _pid_to_korns(r["pid"])
        train = _fmt_pm_sci10(r["mse_train_mean"], r["mse_train_std"])
        test  = _fmt_pm_sci10(r["mse_test_mean"],  r["mse_test_std"])
        nc    = _fmt_pm_sci10(r["node_count_mean"], r["node_count_std"])
        rows.append([dataset, train, test, nc])

    print("\n" + "=" * 120)
    print(f"FILE: {path} | ALGO: {algo_name}")
    print("=" * 120)
    _print_table(rows, headers=["Dataset", "Train MSE", "Test MSE", "NC"])



FILE: korns_pysr_benchmark_results.csv | ALGO: pysr
Dataset   Train MSE          Test MSE           NC             
--------  -----------------  -----------------  ---------------
Korns-1   (1.4+-2.5)*10^-12  (1.4+-2.4)*10^-12  (5.0+-0.0)*10^0
Korns-10  (1.3+-0.2)*10^2    (2.8+-3.5)*10^3    (1.6+-0.5)*10^1
Korns-11  (6.5+-6.4)*10^-9   (6.1+-6.2)*10^-9   (1.0+-0.0)*10^1
Korns-12  (8.0+-4.5)*10^-1   (8.8+-4.9)*10^-1   (1.2+-0.1)*10^1
Korns-13  (3.5+-2.0)*10^2    (3.8+-2.3)*10^2    (1.3+-0.5)*10^1
Korns-14  (2.3+-0.0)*10^-10  (1.8+-0.0)*10^-10  (1.9+-0.0)*10^1
Korns-15  (1.5+-1.9)*10^-11  (1.2+-1.4)*10^-11  (1.8+-0.0)*10^1
Korns-2   (4.5+-6.2)*10^-13  (4.7+-6.4)*10^-13  (1.3+-0.2)*10^1
Korns-3   (5.7+-0.1)*10^-14  (4.1+-0.2)*10^-14  (2.0+-0.1)*10^1
Korns-4   (0.5+-1.0)*10^-16  (0.5+-1.0)*10^-16  (7.6+-3.6)*10^0
Korns-5   (7.4+-0.0)*10^-32  (1.1+-0.0)*10^-31  (6.0+-0.0)*10^0
Korns-6   (1.1+-1.6)*10^-15  (1.1+-1.6)*10^-15  (7.0+-0.0)*10^0
Korns-7   (2.8+-3.4)*10^-9   (2.7+-3.3)*10^-9   (1.